In [3]:
import pandas as pd
import numpy as np 

import re 
import nltk 
from nltk.tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer 
import spacy
nlp = spacy.load("en_core_web_sm")
from spacy.lang.en.stop_words import STOP_WORDS 
from sklearn.preprocessing import LabelEncoder, OneHotEncoder 
from sklearn.feature_extraction.text import TfidfVectorizer 
#from gensim.models import Word2Vec


import warnings
warnings.filterwarnings("ignore") 

In [5]:
train = pd.read_csv("TRAINING.csv")

In [7]:
train.head()

,comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [9]:
train.shape

(37249, 2)

In [11]:
train.isna().sum()

comment     100
category      0
dtype: int64

In [13]:
train.dropna(inplace = True)

## Removing any punctuations

In [16]:
train["cleaned_comment"] = train["comment"].fillna("").astype(str).apply(
    lambda text: re.sub(r'[^A-Za-z0-9]+', ' ', text.lower())
)

## Tokenization

In [19]:
train["comment_token"] = train["cleaned_comment"].apply(word_tokenize)

## Stop Word Removal

In [21]:
train.head()

,comment,category,cleaned_comment,comment_token
0,family mormon have never tried explain them t...,1,family mormon have never tried explain them t...,"[family, mormon, have, never, tried, explain, ..."
1,buddhism has very much lot compatible with chr...,1,buddhism has very much lot compatible with chr...,"[buddhism, has, very, much, lot, compatible, w..."
2,seriously don say thing first all they won get...,-1,seriously don say thing first all they won get...,"[seriously, don, say, thing, first, all, they,..."
3,what you have learned yours and only yours wha...,0,what you have learned yours and only yours wha...,"[what, you, have, learned, yours, and, only, y..."
4,for your own benefit you may want read living ...,1,for your own benefit you may want read living ...,"[for, your, own, benefit, you, may, want, read..."


In [22]:
from spacy.lang.en.stop_words import STOP_WORDS
print("The stop words length: ", len(STOP_WORDS))
for sw in STOP_WORDS:
    print(sw, end = " | ")

The stop words length:  326
over | hundred | of | next | used | something | elsewhere | even | thence | five | show | seemed | anything | themselves | had | into | very | those | due | bottom | the | about | others | each | thru | eleven | itself | side | sometimes | empty | ‘s | name | back | n’t | him | he | really | four | wherein | will | because | could | toward | at | be | first | you | noone | twenty | alone | enough | more | his | it | become | too | their | third | move | take | they | were | otherwise | besides | six | above | beyond | whereupon | fifty | within | beforehand | front | although | however | my | down | twelve | whoever | though | whither | ourselves | still | nowhere | unless | me | another | just | do | namely | around | yourselves | 'd | mostly | make | so | least | 'll | such | therefore | then | please | say | towards | does | and | but | one | behind | none | top | out | throughout | might | various | between | beside | few | several | would | i | perhaps 

In [23]:
spacy_stopwords = nlp.Defaults.stop_words

# Assuming your tokens are in a column like this: ['this', 'is', 'an', 'example']
def remove_stopwords(tokens):
    return [token for token in tokens if token.lower() not in spacy_stopwords]

train["Lemmatized_comment"] = train["comment_token"].apply(remove_stopwords)

## Lemmatization

In [25]:
train

,comment,category,cleaned_comment,comment_token,Lemmatized_comment
0,family mormon have never tried explain them t...,1,family mormon have never tried explain them t...,"[family, mormon, have, never, tried, explain, ...","[family, mormon, tried, explain, stare, puzzle..."
1,buddhism has very much lot compatible with chr...,1,buddhism has very much lot compatible with chr...,"[buddhism, has, very, much, lot, compatible, w...","[buddhism, lot, compatible, christianity, espe..."
2,seriously don say thing first all they won get...,-1,seriously don say thing first all they won get...,"[seriously, don, say, thing, first, all, they,...","[seriously, don, thing, won, complex, explain,..."
3,what you have learned yours and only yours wha...,0,what you have learned yours and only yours wha...,"[what, you, have, learned, yours, and, only, y...","[learned, want, teach, different, focus, goal,..."
4,for your own benefit you may want read living ...,1,for your own benefit you may want read living ...,"[for, your, own, benefit, you, may, want, read...","[benefit, want, read, living, buddha, living, ..."
...,...,...,...,...,...
37244,jesus,0,jesus,[jesus],[jesus]
37245,kya bhai pure saal chutiya banaya modi aur jab...,1,kya bhai pure saal chutiya banaya modi aur jab...,"[kya, bhai, pure, saal, chutiya, banaya, modi,...","[kya, bhai, pure, saal, chutiya, banaya, modi,..."
37246,downvote karna tha par upvote hogaya,0,downvote karna tha par upvote hogaya,"[downvote, karna, tha, par, upvote, hogaya]","[downvote, karna, tha, par, upvote, hogaya]"
37247,haha nice,1,haha nice,"[haha, nice]","[haha, nice]"


## FULL PIPELINE using SPACY

```python
# Import spaCy and load the English language model
import spacy
nlp = spacy.load("en_core_web_sm")

# Define preprocessing function using spaCy
def spacy_preprocess(text):
    # Convert text to lowercase, process with spaCy
    doc = nlp(str(text).lower())
    # Return lemmatized tokens, excluding stopwords and non-alphabetic tokens
    return [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]

# Apply the preprocessing to the 'comment' column
train["processed_comment"] = train["comment"].apply(spacy_preprocess)
```


## Word Embedding/ Vectorization

In [44]:
train["joined_text"] = train["Lemmatized_comment"].apply(lambda tokens: " ".join(tokens))

In [54]:
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 3),
    sublinear_tf=True
)

train["Vectors"] = vectorizer.fit_transform(train["joined_text"])


TypeError: sparse array length is ambiguous; use getnnz() or shape[0]